# Aula 03 — laboratório

## 1. Conectar ao cluster

In [ ]:
import socket
import time

from pyspark.sql import SparkSession
from pyspark.sql import functions as sf

# O IP deste container na rede do compose. Os workers precisam alcancar
# ESTE endereco para devolver resultado ao driver.
meu_ip = socket.gethostbyname(socket.gethostname())
print("IP deste container (o driver):", meu_ip)

spark = (SparkSession.builder
         .appName("aula03")
         # Nao e mais local[*]: e o endereco do cluster manager.
         .master("spark://spark-master:7077")
         # Sem esta linha, os executores nao conseguem voltar ao driver.
         .config("spark.driver.host", meu_ip)
         # Cada executor com 1 core e 512m: 4 executores nos 4 slots.
         .config("spark.executor.cores", "1")
         .config("spark.executor.memory", "512m")
         .config("spark.cores.max", "4")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

print("master     :", spark.sparkContext.master)
print("paralelismo logo apos getOrCreate():",
      spark.sparkContext.defaultParallelism)

In [ ]:
# Espera os executores registrarem antes de medir capacidade.
for _ in range(30):
    infos = spark.sparkContext.statusTracker().getExecutorInfos()
    if len(infos) > 1:
        break
    time.sleep(1)

# A lista inclui o driver, por isso o -1.
print("executores registrados:", len(infos) - 1)
print("paralelismo agora     :", spark.sparkContext.defaultParallelism)
print()
for i in infos:
    print("  ", i.host)

## 2. Uma partição é uma tarefa

In [ ]:
# 8 particoes explicitas.
df = spark.range(0, 10_000_000, numPartitions=8)
print("particoes:", df.rdd.getNumPartitions())

inicio = time.time()
print("linhas   :", df.count())
print("tempo    : %.2f s" % (time.time() - inicio))

## 3. Um shuffle é um stage a mais

In [ ]:
# groupBy exige shuffle: as linhas de uma mesma chave precisam
# acabar na mesma particao.
agrupado = (spark.range(0, 2_000_000, numPartitions=8)
            .withColumn("chave", sf.col("id") % 10)
            .groupBy("chave")
            .count())

inicio = time.time()
agrupado.collect()
print("tempo: %.2f s" % (time.time() - inicio))

In [ ]:
# O default que decide as tarefas DEPOIS do shuffle.
print("spark.sql.shuffle.partitions =",
      spark.conf.get("spark.sql.shuffle.partitions"))

# Mas o que SOBROU na saida?
print("particoes de saida do groupBy:", agrupado.rdd.getNumPartitions())

In [ ]:
print("AQE ligado?          ", spark.conf.get("spark.sql.adaptive.enabled"))
print("coalesce automatico? ",
      spark.conf.get("spark.sql.adaptive.coalescePartitions.enabled"))
print()

# O mesmo groupBy, agora com o AQE desligado.
spark.conf.set("spark.sql.adaptive.enabled", "false")

sem_aqe = (spark.range(0, 2_000_000, numPartitions=8)
           .withColumn("chave", sf.col("id") % 10)
           .groupBy("chave")
           .count())
sem_aqe.collect()

print("particoes de saida SEM AQE:", sem_aqe.rdd.getNumPartitions())

# Religue: o resto do notebook conta com o comportamento padrao.
spark.conf.set("spark.sql.adaptive.enabled", "true")

## 4. Onde o `print` vai parar

In [ ]:
def mostrar(linha):
    # Este print roda NO EXECUTOR, nao aqui.
    print("[EXECUTOR] processando id=%d" % linha["id"])


# foreach roda a funcao em cada linha, nos executores.
spark.range(0, 4, numPartitions=2).foreach(mostrar)

print("--- acabou o foreach. Viu algum [EXECUTOR] acima? ---")

## 5. Quebrando o endereço do driver de propósito

In [ ]:
# Encerre a sessao boa antes de criar a quebrada: so existe uma
# SparkSession por processo.
spark.stop()
print("sessao anterior encerrada")

In [ ]:
import threading

quebrada = (SparkSession.builder
            .appName("aula03-driver-inalcancavel")
            # 127.0.0.1 do ponto de vista do EXECUTOR e o proprio worker.
            .config("spark.driver.host", "127.0.0.1")
            .config("spark.executor.cores", "1")
            .config("spark.executor.memory", "512m")
            .config("spark.cores.max", "2")
            .master("spark://spark-master:7077")
            .getOrCreate())
sc = quebrada.sparkContext
sc.setLogLevel("ERROR")

print("sessao criada. Repare que CRIAR funcionou:")
print("  o driver registrou-se no master sem problema.")
print("  o que vai falhar e o executor tentando voltar.")
print()

# Um grupo de jobs nomeado, para poder CANCELAR depois.
#
# Isto nao e enfeite: o job abaixo nao falha nem termina sozinho, e sem
# um cancelamento explicito esta celula ficaria pendurada para sempre.
# Rodamos num thread e cancelamos o grupo quando o prazo estoura.
sc.setJobGroup("teste-endereco", "job que nao vai completar")

saida = {}


def rodar():
    try:
        saida["n"] = quebrada.range(0, 1000, numPartitions=2).count()
    except Exception as erro:
        saida["erro"] = type(erro).__name__


tarefa = threading.Thread(target=rodar, daemon=True)
inicio = time.time()
tarefa.start()
tarefa.join(timeout=30)

if tarefa.is_alive():
    print("30 s e NENHUMA tarefa executou.")
    print("  nao houve excecao, nao houve falha: o job esta pendurado.")
    print("  cancelando o grupo de jobs...")
    sc.cancelJobGroup("teste-endereco")
    tarefa.join(timeout=15)
    print("  cancelado apos %.0f s" % (time.time() - inicio))
elif "erro" in saida:
    print("falhou apos %.1f s: %s" % (time.time() - inicio, saida["erro"]))
else:
    print("completou em %.1f s (inesperado)" % (time.time() - inicio))

quebrada.stop()
print()
print("sessao encerrada - os slots do cluster foram liberados")

## 6. Isolamento entre aplicações

In [ ]:
s1 = (SparkSession.builder
      .appName("aula03-app-1")
      .master("spark://spark-master:7077")
      .config("spark.driver.host", meu_ip)
      .config("spark.executor.cores", "1")
      .config("spark.executor.memory", "512m")
      .config("spark.cores.max", "2")
      .getOrCreate())
s1.sparkContext.setLogLevel("ERROR")

base = s1.range(0, 2_000_000).selectExpr("id", "id * 2 as dobro").cache()
base.count()                      # materializa o cache

print("app 1 - id da aplicacao:", s1.sparkContext.applicationId)
print("app 1 - em cache?      ", base.is_cached)
print("app 1 - cache em memoria, visivel na aba Storage da UI")

s1.stop()
print()
print("app 1 encerrada: os executores dela morreram, e o cache com eles.")

In [ ]:
s2 = (SparkSession.builder
      .appName("aula03-app-2")
      .master("spark://spark-master:7077")
      .config("spark.driver.host", meu_ip)
      .config("spark.executor.cores", "1")
      .config("spark.executor.memory", "512m")
      .config("spark.cores.max", "2")
      .getOrCreate())
s2.sparkContext.setLogLevel("ERROR")

print("app 2 - id da aplicacao:", s2.sparkContext.applicationId)
print("app 2 - id diferente: e outra aplicacao, com outros executores")
print()

# Nao ha como pedir "o cache da app 1": ele nao existe mais.
# A unica forma de reaproveitar trabalho entre aplicacoes e persistir
# em armazenamento.
inicio = time.time()
s2.range(0, 2_000_000).selectExpr("id", "id * 2 as dobro").count()
print("app 2 recomputou do zero em %.2f s" % (time.time() - inicio))

s2.stop()